# 🎬 Jadi Apa — 1-Click Cloud Render Engine (Google Colab + Blender 5.2.2, v3.0)

**Prinsip Zero-Manual & Satu Pintu:** 100% proses jalan di cloud tanpa laptop kencang, dengan arsitektur modular 3 lapisan (`core/` → `scene_builder.py` → `build_and_render.py`).
- **Maskot Permanen MPFB2 — Mas Dimas v3.0 (*Unified Stylized Semi-Matte* `roughness 0.55–0.65`, 100% Nyeker):** `mas_dimas_ready.blend` otomatis dimuat dari Drive `02_assets/models_3d/`.
- **5 Arketipe Kamera 3D Micro-POV** (`insect_ground`, `aerial_erratic`, `fluid_float`, `static_object_pov`, `human_first_person`) + Motion Blur (`shutter=0.40`)
- **Properti Poly Haven CC0 + AgX + Macro DOF + `post.py`** (soft lens, bloom glow, vignette, film grain)
- **Audio-First Gemini 3.8 Flash TTS (`Puck`, cached `SKIP_EXISTING_VO=True`) + `faster-whisper` Karaoke `.ass` (`1 / VO_SPEED` + anti-hallucination) + `15ms` Micro-Fade + Dynamic Room Tone (`-32 dB`)**
- **Auto-Resume per Frame** (> 10 KB dilewati) + **Auto Error Logger (`error_report.txt`)** tersimpan otomatis ke Google Drive jika terjadi error runtime!

### Pilihan Mode (`RENDER_MODE`):
- **`storyboard` (Adaptive Multi-Keyframe Fast Storyboard):** `270×480` per panel, `4 samples + OIDN` → `storyboard_sheet_1.png`, `storyboard_sheet_2.png`, dst. **Wajib `ACC STORYBOARD` sebelum render `final`!**
- **`preview` (T4 GPU):** Full 24fps `540×960` (16 samples) untuk QC opsional.
- **`final` (T4 GPU — Default):** Full 24fps `1080×1920` (24 samples + OIDN + Motion Blur + `post.py`) siap upload YouTube Shorts / TikTok / Reels.

In [ ]:
#@title 1. Konfigurasi Episode, Mount Google Drive & Auto Error Logger (error_report.txt) { run: "auto" }
import datetime
import os
import shutil
import subprocess
import traceback
from collections import deque
from pathlib import Path

REPO_URL = "https://github.com/ruangcermat/channel-micro-pov.git" #@param {type:"string"}
REPO_BRANCH = "main" #@param {type:"string"}
EPISODE_SPEC = "04_episodes/ep001_kecoak_dapur/ep001_spec.json" #@param {type:"string"}
RENDER_MODE = "final" #@param ["storyboard", "motion_check", "preview", "final"]
RENDER_ENGINE = "CYCLES" #@param ["CYCLES", "EEVEE", "AUTO"]
SINGLE_SCENE = 0 #@param {type:"integer"}
TTS_VOICE = "Puck" #@param ["Puck", "Charon", "Fenrir", "Kore", "Aoede", "Enceladus"]
VO_SPEED = 1.08 #@param {type:"number"}
SKIP_EXISTING_VO = True #@param {type:"boolean"}
SAVE_PACKED_BLEND = False #@param {type:"boolean"}

# 1. Mount Google Drive untuk cache aset & penyimpanan output permanen
from google.colab import drive, userdata
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/channel-micro-pov")
ASSETS_DIR = DRIVE_ROOT / "02_assets"
EPISODE_SLUG = Path(EPISODE_SPEC).parent.name
WORK_DIR = DRIVE_ROOT / "04_episodes" / EPISODE_SLUG

ASSETS_DIR.mkdir(parents=True, exist_ok=True)
(ASSETS_DIR / "models_3d").mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

ERROR_REPORT_PATH = WORK_DIR / "error_report.txt"
_LAST_LOG_TAIL = deque(maxlen=150)

def save_error_report(step_name: str, exc: BaseException):
    """Simpan seluruh traceback & 150 baris log terakhir ke error_report.txt di Google Drive."""
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    tb_text = traceback.format_exc()
    tail_text = "".join(_LAST_LOG_TAIL)
    report = (
        f"=== JADI APA PIPELINE ERROR REPORT ===\n"
        f"Timestamp     : {ts}\n"
        f"Environment   : Google Colab\n"
        f"Step          : {step_name}\n"
        f"Episode Spec  : {EPISODE_SPEC}\n"
        f"Render Mode   : {RENDER_MODE} | Engine: {RENDER_ENGINE} | Scene: {SINGLE_SCENE or 'ALL'}\n"
        f"Work Dir      : {WORK_DIR}\n"
        f"Exception     : {type(exc).__name__}: {exc}\n\n"
        f"--- FULL TRACEBACK ---\n{tb_text}\n"
        f"--- LAST 150 LINES OF OUTPUT ---\n{tail_text}\n"
    )
    for dest in (ERROR_REPORT_PATH, Path("/content/error_report.txt")):
        try:
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_text(report, encoding="utf-8")
        except Exception as write_err:
            print(f"⚠️ Gagal menulis {dest}: {write_err}")
    print("\n" + "=" * 72)
    print(f"❌ ERROR PADA LANGKAH: {step_name}")
    print(f"📄 Traceback lengkap otomatis disimpan ke: {ERROR_REPORT_PATH}")
    print("   (Buka & copy isi error_report.txt di Google Drive tanpa perlu screenshot!)")
    print("=" * 72)
    print(tb_text)

def run_step(cmd, step_name: str = ""):
    """Jalankan subprocess dengan output real-time + otomatis simpan error_report.txt jika gagal."""
    label = step_name or (" ".join(str(x) for x in cmd[:3]))
    _LAST_LOG_TAIL.clear()
    try:
        env = os.environ.copy()
        env["PYTHONUNBUFFERED"] = "1"
        proc = subprocess.Popen(
            cmd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            _LAST_LOG_TAIL.append(line)
            print(line, end="", flush=True)
        rc = proc.wait()
        if rc != 0:
            raise subprocess.CalledProcessError(rc, cmd)
    except Exception as exc:
        save_error_report(label, exc)
        raise

def run_guarded(step_name: str, fn):
    """Wrapper try...except untuk blok Python agar semua runtime error otomatis tercatat di error_report.txt."""
    try:
        return fn()
    except Exception as exc:
        save_error_report(step_name, exc)
        raise

# 2a. Auto-detect Master Maskot Permanen Mas Dimas (mas_dimas_ready.blend)
MAS_DIMAS_CACHE = ASSETS_DIR / "models_3d" / "mas_dimas_ready.blend"
if MAS_DIMAS_CACHE.exists():
    os.environ["MICRO_POV_MAS_DIMAS_BLEND"] = str(MAS_DIMAS_CACHE)
    print(f"✅ Master Mas Dimas siap ({MAS_DIMAS_CACHE.stat().st_size / 1e6:.1f} MB): {MAS_DIMAS_CACHE}")
else:
    print("ℹ️ mas_dimas_ready.blend belum ada di Drive; akan otomatis dibuild dari family_ready.blend jika tersedia.")

# 2b. Auto-sync MPFB2 family_ready.blend dari Google Drive ke cache proyek (fallback/source)
FAMILY_CACHE = ASSETS_DIR / "models_3d" / "family_ready.blend"
FAMILY_DRIVE_SOURCES = [
    Path("/content/drive/MyDrive/Skill referensi/Zack Blender Pack/Zack-Blender-Pack/zack-3d-short/ready/family_ready.blend"),
    DRIVE_ROOT / "05_skills_ref/Zack-Blender-Pack/zack-3d-short/ready/family_ready.blend",
]
if not FAMILY_CACHE.exists():
    for src in FAMILY_DRIVE_SOURCES:
        if src.exists():
            print(f"📦 Menyalin MPFB2 family_ready.blend ke cache proyek: {FAMILY_CACHE}")
            shutil.copy2(src, FAMILY_CACHE)
            break
if FAMILY_CACHE.exists():
    os.environ["MICRO_POV_FAMILY_BLEND"] = str(FAMILY_CACHE)
    print(f"✅ MPFB2 family_ready.blend siap ({FAMILY_CACHE.stat().st_size / 1e6:.1f} MB)")
elif not MAS_DIMAS_CACHE.exists():
    print("⚠️ Baik mas_dimas_ready.blend maupun family_ready.blend tidak ditemukan di Drive!")

# 3. Baca Secrets opsional (GEMINI_API_KEY & GITHUB_TOKEN jika repo private)
GITHUB_TOKEN = ""
for secret_name in ("GEMINI_API_KEY", "GITHUB_TOKEN"):
    try:
        val = userdata.get(secret_name)
        if val and val.strip():
            if secret_name == "GITHUB_TOKEN":
                GITHUB_TOKEN = val.strip()
            else:
                os.environ[secret_name] = val.strip()
            print(f"✅ Secret {secret_name} dimuat dari Colab Secrets.")
    except Exception:
        pass

os.environ["MICRO_POV_ASSETS_DIR"] = str(ASSETS_DIR)
os.environ["MICRO_POV_WORK_DIR"] = str(WORK_DIR)

print(f"📁 Drive Assets Cache : {ASSETS_DIR}")
print(f"📁 Episode Output Dir : {WORK_DIR}")
print(f"🛡️ Error Report Auto  : {ERROR_REPORT_PATH}")
print(f"🎞️ Mode: {RENDER_MODE} | Engine: {RENDER_ENGINE} | Scene: {SINGLE_SCENE or 'ALL'} | VO Speed: {VO_SPEED}x | 24fps")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || echo "ℹ️ Mode CPU aktif (gunakan T4 GPU untuk motion_check/preview/final)."

In [ ]:
#@title 2. Sync Repository GitHub & Install Dependensi Pipeline
def _step_sync_repo():
    AUTH_REPO_URL = REPO_URL
    if GITHUB_TOKEN and REPO_URL.startswith("https://"):
        AUTH_REPO_URL = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@", 1)

    repo_dir = Path("/content/channel-micro-pov")
    if (repo_dir / ".git").exists():
        print("🔄 Pulling update terbaru dari GitHub...")
        run_step(["git", "-C", str(repo_dir), "remote", "set-url", "origin", AUTH_REPO_URL], "git remote")
        run_step(["git", "-C", str(repo_dir), "fetch", "origin"], "git fetch")
        run_step(["git", "-C", str(repo_dir), "checkout", REPO_BRANCH], "git checkout")
        run_step(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", REPO_BRANCH], "git pull")
    else:
        print("📥 Cloning repository dari GitHub...")
        run_step(["git", "clone", "-b", REPO_BRANCH, AUTH_REPO_URL, str(repo_dir)], "git clone")

    run_step(
        ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core fonts-liberation libxrender1 libxxf86vm1 libxfixes3 libxi6 libxkbcommon0 libsm6 libgl1 libegl1 libglu1-mesa > /dev/null"],
        "apt-get install",
    )
    run_step(["pip", "install", "-q", "--upgrade", "-r", str(repo_dir / "03_pipeline/requirements.txt")], "pip install")
    print("✅ Repo & dependensi Python/faster-whisper/FFmpeg/Font/GL siap!")
    return repo_dir

REPO_DIR = run_guarded("Cell 2: Sync Repo & Dependencies", _step_sync_repo)

In [ ]:
#@title 3. Install Blender 5.2.2 LTS Linux Headless
def _step_install_blender():
    blender_version = "5.2.2"
    blender_tar = f"blender-{blender_version}-linux-x64.tar.xz"
    blender_url = f"https://download.blender.org/release/Blender5.2/{blender_tar}"
    blender_bin = Path("/opt/blender52/blender")

    if not blender_bin.exists():
        print(f"⬇️ Mengunduh Blender {blender_version} LTS Linux Headless...")
        run_step(["wget", "-q", "--show-progress", blender_url, "-O", f"/tmp/{blender_tar}"], "wget Blender 5.2.2")
        run_step(["mkdir", "-p", "/opt/blender52"], "mkdir /opt/blender52")
        run_step(["tar", "-xf", f"/tmp/{blender_tar}", "-C", "/opt/blender52", "--strip-components=1"], "extract Blender")
        run_step(["rm", "-f", f"/tmp/{blender_tar}"], "cleanup tarball")
    else:
        print("✅ Blender 5.2.2 headless sudah terpasang di /opt/blender52")

    run_step([str(blender_bin), "--version"], "blender --version")
    return blender_bin

BLENDER_BIN = run_guarded("Cell 3: Install Blender 5.2.2", _step_install_blender)

In [ ]:
#@title 4. Auto-Fetch Properti Poly Haven CC0 + Tekstur PBR + Sync Cache Mas Dimas
SPEC_PATH = REPO_DIR / EPISODE_SPEC
run_step([
    "python3", str(REPO_DIR / "03_pipeline/scripts/fetch_assets.py"),
    "--spec", str(SPEC_PATH),
    "--assets-dir", str(ASSETS_DIR),
], "Cell 4: fetch_assets.py")

In [ ]:
#@title 5. Audio-First Voiceover — Cache Puck Drive (SKIP_EXISTING_VO=True, 24fps, 1.08x)
import json
from IPython.display import Audio, display

def _step_tts():
    tts_cmd = [
        "python3",
        str(REPO_DIR / "03_pipeline/scripts/generate_tts.py"),
        "--spec", str(SPEC_PATH),
        "--work-dir", str(WORK_DIR),
        "--voice", TTS_VOICE,
        "--fps", "24",
        "--pad-sec", "0.06",
        "--vo-speed", str(VO_SPEED),
    ]
    if SKIP_EXISTING_VO:
        tts_cmd.append("--skip-existing")
    if SINGLE_SCENE > 0:
        tts_cmd.extend(["--scene", str(SINGLE_SCENE)])

    run_step(tts_cmd, "Cell 5: generate_tts.py")

    manifest_path = WORK_DIR / "timing_manifest.json"
    m = json.loads(manifest_path.read_text(encoding="utf-8"))
    print(f"\n⏱️ Total durasi Audio-First ({m.get('vo_speed', 1.0)}x): {m['total_duration_sec']:.2f}s ({m['total_frames']} frames @ {m['fps']}fps)")
    for sc in m["scenes"]:
        print(
            f"  Scene {sc['scene_id']:02d}: {sc['start_sec']:5.2f}s - {sc['end_sec']:5.2f}s "
            f"(VO {sc['audio_duration_sec']:.2f}s, {sc['frame_count']}f) | {sc['vo_text'][:52]}..."
        )
    first_wav = Path(m["scenes"][0]["wav_path"])
    if first_wav.exists():
        print("\n🔊 Preview VO Scene 01:")
        display(Audio(filename=str(first_wav)))
    return m

manifest = run_guarded("Cell 5: Audio-First VO", _step_tts)

In [ ]:
#@title 6. Auto-Prepare SFX & BGM CC0
run_step([
    "python3", str(REPO_DIR / "03_pipeline/scripts/prepare_audio.py"),
    "--spec", str(SPEC_PATH),
    "--assets-dir", str(ASSETS_DIR),
], "Cell 6: prepare_audio.py")

In [ ]:
#@title 7. Blender 5.2.2 Modular Runner (Mas Dimas + Core POV Engine + AgX + post.py + Auto-Resume)
from IPython.display import Image as IPyImage, display

def _step_render():
    render_cmd = [
        str(BLENDER_BIN),
        "--background",
        "--python", str(REPO_DIR / "03_pipeline/scripts/build_and_render.py"),
        "--",
        "--spec", str(SPEC_PATH),
        "--timing", str(WORK_DIR / "timing_manifest.json"),
        "--assets-dir", str(ASSETS_DIR),
        "--work-dir", str(WORK_DIR),
        "--mode", RENDER_MODE,
        "--engine", RENDER_ENGINE,
    ]
    if SAVE_PACKED_BLEND:
        render_cmd.append("--save-blend")
    if SINGLE_SCENE > 0:
        render_cmd.extend(["--scene", str(SINGLE_SCENE)])

    print("🚀 Menjalankan Blender 5.2.2:", " ".join(render_cmd))
    run_step(render_cmd, f"Cell 7: Blender Render ({RENDER_MODE})")

    if RENDER_MODE == "storyboard":
        sheets = sorted(WORK_DIR.glob("storyboard_sheet_*.png"))
        if not sheets and (WORK_DIR / "storyboard_sheet.png").exists():
            sheets = [WORK_DIR / "storyboard_sheet.png"]
        for sp in sheets:
            print(f"\n🎬 [GATE 1] Adaptive Storyboard Sheet: {sp}")
            display(IPyImage(filename=str(sp)))

run_guarded("Cell 7: Blender 3D Render", _step_render)

In [ ]:
#@title 8. Subtitle Karaoke faster-whisper (.ass 9:16, 1/VO_SPEED) + 4-Layer Audio (-14 LUFS)
run_step([
    "python3", str(REPO_DIR / "03_pipeline/scripts/compose_final.py"),
    "--spec", str(SPEC_PATH),
    "--timing", str(WORK_DIR / "timing_manifest.json"),
    "--assets-dir", str(ASSETS_DIR),
    "--work-dir", str(WORK_DIR),
    "--mode", RENDER_MODE,
], "Cell 8: compose_final.py")

In [ ]:
#@title 9. Player Preview Video Langsung di Colab & Lokasi File Google Drive
import base64
from IPython.display import HTML, display

def _step_preview():
    ep_id = manifest["episode_id"].lower()
    if RENDER_MODE == "final":
        out_name = f"final_{ep_id}.mp4"
    elif RENDER_MODE == "storyboard":
        out_name = f"animatic_{ep_id}.mp4"
    elif RENDER_MODE == "motion_check":
        out_name = f"motion_check_{ep_id}.mp4"
    else:
        out_name = f"preview_{ep_id}.mp4"
    video_path = WORK_DIR / out_name

    assert video_path.exists(), f"File video tidak ditemukan: {video_path}"
    size_mb = video_path.stat().st_size / (1024 * 1024)
    print("🎉 SELESAI! Video tersimpan permanen di Google Drive:")
    print(f"   📂 {video_path} ({size_mb:.2f} MB)")

    if size_mb <= 35.0:
        b64 = base64.b64encode(video_path.read_bytes()).decode("ascii")
        html_player = (
            '<div style="display:flex;justify-content:center;padding:12px;background:#111;border-radius:12px;">'
            '<video width="360" height="640" controls playsinline '
            'style="border-radius:10px;box-shadow:0 8px 24px rgba(0,0,0,0.5);">'
            f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
            "</video></div>"
        )
        display(HTML(html_player))
    else:
        print("ℹ️ Ukuran file > 35 MB; silakan putar langsung dari Google Drive kamu.")

run_guarded("Cell 9: Preview & Verify Output", _step_preview)